# Weather Trend Forecasting - Data Cleaning and Preprocessing

In [100]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Load the dataset from the data folder.
# This step is needed so the notebook can work with the weather observations.
project_root = Path.cwd()
data_path = project_root / "data" / "GlobalWeatherRepository.csv"
if not data_path.exists():
    data_path = project_root / ".." / "data" / "GlobalWeatherRepository.csv"

raw_df = pd.read_csv(data_path)

# Keep a copy of the original data for reference.
# This protects the source data from accidental changes during cleaning.
df = raw_df.copy()

# Inspect the dataset shape, column names, and data types.
# This step helps us understand the structure before preprocessing.
print("Rows before cleaning:", df.shape[0])
print("Columns before cleaning:", df.shape[1])
print("\nFirst rows:")
print(df.head(3))
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Rows before cleaning: 152412
Columns before cleaning: 41

First rows:
       country location_name  latitude  longitude        timezone  \
0  Afghanistan         Kabul     34.52      69.18      Asia/Kabul   
1      Albania        Tirana     41.33      19.82   Europe/Tirane   
2      Algeria       Algiers     36.76       3.05  Africa/Algiers   

   last_updated_epoch      last_updated  temperature_celsius  \
0          1715849100  2024-05-16 13:15                 26.6   
1          1715849100  2024-05-16 10:45                 19.0   
2          1715849100  2024-05-16 09:45                 23.0   

   temperature_fahrenheit condition_text  ...  air_quality_PM2.5  \
0                    79.8  Partly Cloudy  ...                8.4   
1                    66.2  Partly cloudy  ...                1.1   
2                    73.4          Sunny  ...               10.4   

   air_quality_PM10  air_quality_us-epa-index air_quality_gb-defra-index  \
0              26.6                         1  

In [101]:
# Convert the timestamp column into datetime format.
# This is required for extracting time-based features later.
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

# Check missing values after the datetime conversion.
# This helps us measure how much repair is needed.
print("Missing values after datetime conversion:")
print(df.isnull().sum())

Missing values after datetime conversion:
country                         0
location_name                   0
latitude                        0
longitude                       0
timezone                        0
last_updated_epoch              0
last_updated                    0
temperature_celsius             0
temperature_fahrenheit          0
condition_text                  0
wind_mph                        0
wind_kph                        0
wind_degree                     0
wind_direction                  0
pressure_mb                     0
pressure_in                     0
precip_mm                       0
precip_in                       0
humidity                        0
cloud                           0
feels_like_celsius              0
feels_like_fahrenheit           0
visibility_km                   0
visibility_miles                0
uv_index                        0
gust_mph                        0
gust_kph                        0
air_quality_Carbon_Monoxide     0
air_qu

In [102]:
# Replace invalid numeric placeholders such as -9999 with missing values.
# This ensures that impossible values do not distort the statistics.
invalid_value_columns = [
    "temperature_celsius",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "humidity",
    "cloud",
    "visibility_km",
    "uv_index",
    "air_quality_Carbon_Monoxide",
    "air_quality_Sulphur_dioxide",
    "air_quality_PM10",
]
for col in invalid_value_columns:
    if col in df.columns:
        df[col] = df[col].replace(-9999, np.nan)

print("Missing values after invalid value replacement:")
print(df[invalid_value_columns].isnull().sum())

Missing values after invalid value replacement:
temperature_celsius            0
pressure_mb                    0
wind_kph                       0
precip_mm                      0
humidity                       0
cloud                          0
visibility_km                  0
uv_index                       0
air_quality_Carbon_Monoxide    1
air_quality_Sulphur_dioxide    1
air_quality_PM10               0
dtype: int64


In [103]:
# Detect outliers using the interquartile range method.
# This helps remove extreme values that can harm forecasting quality.
outlier_features = ["temperature_celsius", "pressure_mb", "wind_kph", "precip_mm"]
outlier_summary = {}

for col in outlier_features:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_summary[col] = int(mask.sum())
        df.loc[mask, col] = np.nan

print("Outliers replaced with NaN:")
print(outlier_summary)
print("\nMissing values after outlier treatment:")
print(df[outlier_features].isnull().sum())

Outliers replaced with NaN:
{'temperature_celsius': 2983, 'pressure_mb': 4241, 'wind_kph': 2546, 'precip_mm': 30637}

Missing values after outlier treatment:
temperature_celsius     2983
pressure_mb             4241
wind_kph                2546
precip_mm              30637
dtype: int64


In [104]:
# Fill remaining missing values with the median for numeric columns.
# This keeps the dataset complete without dropping rows.
numeric_columns = [
    "latitude",
    "longitude",
    "humidity",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "cloud",
    "visibility_km",
    "uv_index",
    "temperature_celsius",
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Create time-based features from the timestamp.
# These help the model capture seasonal and daily patterns.
df["year"] = df["last_updated"].dt.year
df["month"] = df["last_updated"].dt.month
df["day"] = df["last_updated"].dt.day
df["hour"] = df["last_updated"].dt.hour

print("Missing values after imputation:")
print(df[numeric_columns + ["year", "month", "day", "hour"]].isnull().sum())
print("\nTime feature preview:")
print(df[["last_updated", "year", "month", "day", "hour"]].head())

Missing values after imputation:
latitude               0
longitude              0
humidity               0
pressure_mb            0
wind_kph               0
precip_mm              0
cloud                  0
visibility_km          0
uv_index               0
temperature_celsius    0
year                   0
month                  0
day                    0
hour                   0
dtype: int64

Time feature preview:
         last_updated  year  month  day  hour
0 2024-05-16 13:15:00  2024      5   16    13
1 2024-05-16 10:45:00  2024      5   16    10
2 2024-05-16 09:45:00  2024      5   16     9
3 2024-05-16 10:45:00  2024      5   16    10
4 2024-05-16 09:45:00  2024      5   16     9


In [105]:
# Select the variables that are most useful for forecasting.
# These columns represent the main physical and temporal signals for the model.
model_features = [
    "latitude",
    "longitude",
    "month",
    "day",
    "hour",
    "humidity",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "cloud",
    "visibility_km",
    "uv_index",
]
target_col = "temperature_celsius"

processed_df = df[model_features + [target_col]].copy()

print("Processed dataframe shape:", processed_df.shape)
print("\nProcessed dataframe preview:")
print(processed_df.head())

Processed dataframe shape: (152412, 13)

Processed dataframe preview:
   latitude  longitude  month  day  hour  humidity  pressure_mb  wind_kph  \
0     34.52      69.18      5   16    13        24       1012.0      13.3   
1     41.33      19.82      5   16    10        94       1012.0      11.2   
2     36.76       3.05      5   16     9        29       1011.0      15.1   
3     42.50       1.52      5   16    10        61       1007.0      11.9   
4     -8.84      13.23      5   16     9        89       1011.0      13.0   

   precip_mm  cloud  visibility_km  uv_index  temperature_celsius  
0        0.0     30           10.0       7.0                 26.6  
1        0.0     75           10.0       5.0                 19.0  
2        0.0      0           10.0       5.0                 23.0  
3        0.0    100            2.0       2.0                  6.3  
4        0.0     50           10.0       8.0                 26.0  


In [107]:
# Standardize the selected numerical features.
# This makes the model less sensitive to the scale of different variables.
scaler = StandardScaler()
processed_df[model_features] = scaler.fit_transform(processed_df[model_features])

print("Scaled dataframe preview:")
print(processed_df.head())
print("\nFeature means after scaling:")
print(processed_df[model_features].mean().round(4))
print("\nFeature standard deviations after scaling:")
print(processed_df[model_features].std().round(4))

Scaled dataframe preview:
   latitude  longitude     month       day      hour  humidity  pressure_mb  \
0  0.626220   0.718982 -0.447209  0.030291  0.486782 -1.815931    -0.343120   
1  0.905322  -0.031415 -0.447209  0.030291 -0.142087  1.144533    -0.343120   
2  0.718024  -0.286362 -0.447209  0.030291 -0.351710 -1.604469    -0.511475   
3  0.953274  -0.309622 -0.447209  0.030291 -0.142087 -0.251114    -1.184893   
4 -1.150857  -0.131600 -0.447209  0.030291 -0.351710  0.933071    -0.511475   

   wind_kph  precip_mm     cloud  visibility_km  uv_index  temperature_celsius  
0  0.138662  -0.324117 -0.279863       0.179548  1.081545                 26.6  
1 -0.142924  -0.324117  1.040225       0.179548  0.512298                 19.0  
2  0.380022  -0.324117 -1.159923       0.179548  0.512298                 23.0  
3 -0.049062  -0.324117  1.773608      -2.797086 -0.341573                  6.3  
4  0.098435  -0.324117  0.306843       0.179548  1.366168                 26.0  

Feature mean

In [ ]:
## EDA - Temperature Time Series Analysis


In [111]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    df["last_updated"],
    df["temperature_celsius"]
)

plt.title("Temperature Over Time")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")

plt.xticks(rotation=45)
plt.grid()

plt.show()


ModuleNotFoundError: No module named 'matplotlib'

In [113]:
## precipitation change over time
plt.figure(figsize=(12, 5))
plt.plot(
    df["last_updated"],
    df["precip_mm"]
)
plt.title("Precipitation Over Time")
plt.xlabel("Date")
plt.ylabel("Precipitation (mm)")

plt.xticks(rotation=45)
plt.grid()

plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Select important numerical weather features
corr_columns = [
    "temperature_celsius",
    "humidity",
    "pressure_mb",
    "wind_kph",
    "precip_mm",
    "cloud",
    "visibility_km",
    "uv_index"
]
corr_matrix = df[corr_columns].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Weather Features Correlation Heatmap")
plt.show()


NameError: name 'plt' is not defined